In [ ]:
# STEP 1: Mount Google Drive

# Import the Google Drive access module from Google Colab
# This allows the program to read files stored in your Google Drive
from google.colab import drive

# Connect (mount) Google Drive so the notebook can access datasets and model files
drive.mount('/content/drive')


# STEP 2: Import libraries

# Import OS library
# Used to work with file paths and folders
import os

# Import NumPy library
# Helps handle numerical data such as image arrays
import numpy as np

# Import glob library
# Used to search for files in a folder using patterns like *.jpg
import glob

# Import function to load a previously trained machine learning model
from tensorflow.keras.models import load_model

# Import image utilities
# These tools help load images and convert them into arrays that the AI model can process
from tensorflow.keras.preprocessing import image


# STEP 3: Define paths

# Define the path where the trained MobileNetV2 model is stored in Google Drive
model_path = '/content/drive/MyDrive/ML_WSSV/ML_WSSV/WSSV/Trained_Model/mobilenetv2_model.h5'

# Define the folder containing shrimp images that need to be analyzed
test_images_path = '/content/drive/MyDrive/ML_WSSV/WSSV/images'


# STEP 4: Load the trained model

# Load the trained machine learning model from the specified location
# This model was previously trained to detect WSSV disease in shrimp
model = load_model(model_path)

# Print confirmation message to show that the model loaded successfully
print(f"✅ Model loaded from: {model_path}")


# STEP 5: Predict function

# Define a function that predicts whether a shrimp image is Healthy or WSSV infected
# The function receives the image location and the trained model
def predict_image_class(image_path, model):

    # Load the image from the given file path
    # Resize the image to 224x224 pixels because the model expects this size
    img = image.load_img(image_path, target_size=(224, 224))

    # Convert the image into a numeric array
    # Machine learning models work with numbers instead of raw images
    img_array = image.img_to_array(img) / 255.0

    # Normalize pixel values by dividing by 255
    # This converts values from range (0–255) to (0–1)
    # Normalization helps the model make more stable predictions

    # Expand dimensions to simulate a batch of images
    # Even when predicting a single image, the model expects a batch format
    img_array = np.expand_dims(img_array, axis=0)

    # Use the trained model to predict the probability of the shrimp being infected
    prediction = model.predict(img_array, verbose=0)[0][0]

    # If the prediction value is greater than 0.5, classify the shrimp as WSSV infected
    # Otherwise classify it as Healthy
    class_label = 'WSSV' if prediction > 0.5 else 'HEALTHY'

    # Calculate confidence score for the prediction
    # If WSSV predicted → use prediction value
    # If Healthy predicted → use inverse probability
    confidence = prediction if prediction > 0.5 else 1 - prediction

    # Return the predicted class and confidence percentage
    return class_label, round(confidence * 100, 2)


# STEP 6: Scan folder for image files

# Define image file formats that the program should look for
image_extensions = ['*.jpg', '*.jpeg', '*.png']

# Create an empty list to store paths of all detected images
all_images = []

# Loop through each image extension type
for ext in image_extensions:

    # Search the image folder and collect all matching image files
    all_images.extend(glob.glob(os.path.join(test_images_path, ext)))

# Print which folder is being scanned
print(f"📁 Scanning folder: {test_images_path}")

# Print how many image files were found
print(f"🧾 Found {len(all_images)} image(s).")


# STEP 7: Predict each image

# Loop through each image found in the folder
for img_path in all_images:

    try:
        # Call the prediction function for the current image
        label, conf = predict_image_class(img_path, model)

        # Print the prediction result
        # Shows the image name, predicted class, and confidence percentage
        print(f"🖼️ {os.path.basename(img_path)} → Predicted: {label} ({conf}% confidence)")

    except Exception as e:
        # If an error occurs (for example corrupted image file),
        # print an error message instead of stopping the program
        print(f"❌ Error with {os.path.basename(img_path)}: {e}")

Mounted at /content/drive


✅ Model loaded from: /content/drive/MyDrive/ML_WSSV/shrimpInfectionDetection/mobilenetv2_model.h5
📁 Scanning folder: /content/drive/MyDrive/ML_WSSV/shrimpInfectionDetection/images
🧾 Found 2 image(s).
🖼️ Screenshot 2023-11-09 at 12.56.38.png → Predicted: HEALTHY (96.91999816894531% confidence)
🖼️ Screenshot 2023-11-09 at 12.57.32.png → Predicted: WSSV (93.43000030517578% confidence)
